In [12]:
import pandas as pd
import os

MASTER_PRIMARY_KEYS = {
    'orders': 'order_id',
    'customers': 'customer_id',
    'products': 'product_id',
    'promotions': 'promo_id',
    'geography': 'zip',
    'returns': 'return_id',
    'reviews': 'review_id',
    'sales': 'Date'
}

def clean_and_export_table(file_name):
    
    table_name = file_name.replace('.csv', '')
    print(f"\n Đang xử lý bảng: {table_name.upper()}...")
    
    try:
        if table_name == 'products':
            df = pd.read_csv('products.csv')
        elif table_name == 'customers':
            df = pd.read_csv('customers.csv')
        elif table_name == 'promotions':
            df = pd.read_csv('promotions.csv')
        elif table_name == 'geography':
            df = pd.read_csv('geography.csv')
        elif table_name == 'orders':
            df = pd.read_csv('orders.csv')
        elif table_name == 'order_items':
            df = pd.read_csv('order_items.csv',dtype={'promo_id': str, 'promo_id_2': str})
        elif table_name == 'payments':
            df = pd.read_csv('payments.csv')
        elif table_name == 'returns':
            df = pd.read_csv('returns.csv')
        elif table_name == 'shipments':
            df = pd.read_csv('shipments.csv')
        elif table_name == 'reviews':
            df = pd.read_csv('reviews.csv')
        elif table_name == 'sales':
            df = pd.read_csv('sales.csv')
        elif table_name == 'web_traffic':
            df = pd.read_csv('web_traffic.csv')
        elif table_name == 'inventory':
            df = pd.read_csv('inventory.csv')
            
        df = df.drop_duplicates()

        if table_name in MASTER_PRIMARY_KEYS:
            pk_col = MASTER_PRIMARY_KEYS[table_name]
            df = df.dropna(subset=[pk_col])
            df = df.drop_duplicates(subset=[pk_col], keep='last')
        
        if table_name == 'products':
            text_cols_proc = ['category', 'segment', 'color']
            for col in text_cols_proc:
                df[col] = df[col].astype(str).str.strip().str.lower()
            df['size'] = df['size'].astype(str).str.strip().str.upper()
            
        elif table_name == 'customers':
            df['signup_date'] = pd.to_datetime(df['signup_date'])
            text_cols_cus = ['city', 'gender', 'age_group', 'acquisition_channel']
            for col in text_cols_cus:
                df[col] = df[col].astype(str).str.strip().str.lower()
            
        elif table_name == 'promotions':
            df['start_date'] = pd.to_datetime(df['start_date'])
            df['end_date'] = pd.to_datetime(df['end_date'])
            df['applicable_category'] = df['applicable_category'].fillna('All Categories')
            text_cols_promo = ['promo_type', 'promo_channel']
            for col in text_cols_promo:
                df[col] = df[col].astype(str).str.strip().str.lower()
                
        elif table_name == 'geography':
            text_cols_geo = ['city', 'region', 'district']
            for col in text_cols_geo:
                df[col] = df[col].astype(str).str.strip().str.title()
            
        elif table_name == 'orders':
            df['order_date'] = pd.to_datetime(df['order_date'])
            df = df.dropna(subset=['order_id'])
            df = df.drop_duplicates(subset=['order_id'], keep='last')
            text_cols_ord = ['order_status', 'payment_method', 'device_type', 'order_source']
            for col in text_cols_ord:
                df[col] = df[col].astype(str).str.strip().str.lower()
            
        elif table_name == 'order_items':
            df['line_revenue'] = df['quantity'] * df['unit_price']
            df = df.groupby(['order_id', 'product_id'], dropna=False).agg({
                'quantity': 'sum',
                'discount_amount': 'sum',        
                'promo_id': 'first',             
                'promo_id_2': 'first',           
                'line_revenue': 'sum'            
            }).reset_index()
            df['unit_price'] = df['line_revenue'] / df['quantity']
            df = df.drop(columns=['line_revenue'])

        elif table_name == 'payments':
            df = df.dropna(subset=['order_id'])
            df = df.groupby('order_id', dropna=False).agg({
                'payment_method': 'first',     
                'payment_value': 'sum',        
                'installments': 'max'          
            }).reset_index()
            df = df[df['payment_value'] >= 0]
            df['payment_method'] = df['payment_method'].astype(str).str.strip().str.lower()
        
        elif table_name == 'returns':
            df['return_date'] = pd.to_datetime(df['return_date'])
            df = df.drop_duplicates(subset=['return_id'], keep='first')   
            
        elif table_name == 'shipments':
            df['ship_date'] = pd.to_datetime(df['ship_date'])
            df['delivery_date'] = pd.to_datetime(df['delivery_date'])
            
        elif table_name == 'reviews':
            df['review_date'] = pd.to_datetime(df['review_date'])
            df['review_title'] = df['review_title'].astype(str).str.strip().str.lower()
            
        elif table_name == 'sales':
            df['Date'] = pd.to_datetime(df['Date'])
            df.columns = df.columns.str.lower()
        
        elif table_name == 'web_traffic':
            df['date'] = pd.to_datetime(df['date'])
            df['traffic_source'] = df['traffic_source'].astype(str).str.strip().str.lower()
            
        elif table_name == 'inventory':
            df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])
            text_cols_inven = ['product_name', 'category', 'segment']
            for col in text_cols_inven:
                df[col] = df[col].astype(str).str.strip().str.lower()
            df['year'] = df['year'].astype(int)
            df['month'] = df['month'].astype(int)
            
        output_file = f"{table_name}_base.csv"
        df.to_csv(output_file, index=False)
        print(f"Xong! Đã lưu thành: {output_file} | Kích thước: {df.shape}")
        
    except Exception as e:
        print(f"Lỗi khi xử lý {table_name}: {e}")
        
list_files = [
    'products.csv', 'customers.csv', 'promotions.csv', 'geography.csv', 'orders.csv', 'order_items.csv', 'payments.csv', 'returns.csv', 
    'shipments.csv', 'reviews.csv', 'sales.csv', 'web_traffic.csv', 'inventory.csv']

for file in list_files:
    if os.path.exists(file):
        clean_and_export_table(file)
    else:
        print(f"Không tìm thấy file: {file}")


 Đang xử lý bảng: PRODUCTS...
Xong! Đã lưu thành: products_base.csv | Kích thước: (2412, 8)

 Đang xử lý bảng: CUSTOMERS...
Xong! Đã lưu thành: customers_base.csv | Kích thước: (121930, 7)

 Đang xử lý bảng: PROMOTIONS...
Xong! Đã lưu thành: promotions_base.csv | Kích thước: (50, 10)

 Đang xử lý bảng: GEOGRAPHY...
Xong! Đã lưu thành: geography_base.csv | Kích thước: (39948, 4)

 Đang xử lý bảng: ORDERS...
Xong! Đã lưu thành: orders_base.csv | Kích thước: (646945, 8)

 Đang xử lý bảng: ORDER_ITEMS...
Xong! Đã lưu thành: order_items_base.csv | Kích thước: (714653, 7)

 Đang xử lý bảng: PAYMENTS...
Xong! Đã lưu thành: payments_base.csv | Kích thước: (646945, 4)

 Đang xử lý bảng: RETURNS...
Xong! Đã lưu thành: returns_base.csv | Kích thước: (39939, 7)

 Đang xử lý bảng: SHIPMENTS...
Xong! Đã lưu thành: shipments_base.csv | Kích thước: (566067, 4)

 Đang xử lý bảng: REVIEWS...
Xong! Đã lưu thành: reviews_base.csv | Kích thước: (113551, 7)

 Đang xử lý bảng: SALES...
Xong! Đã lưu thành: s